In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import pandas as pd
import math

# Load the dataset
file_path = "C:\\Users\\Jack\\Desktop\\foodRecipeAndInteractions\\RAW_recipes.csv"
df = pd.read_csv(file_path)

# Extract relevant fields and handle missing data
df = df[['name', 'steps', 'description']].fillna('')

# Combine 'name', 'description', and 'steps' into a single text field for training
df['text'] = df['name'] + ' ' + df['description'] + ' ' + df['steps'].apply(lambda x: ' '.join(eval(x)) if isinstance(x, str) else '')

# Convert all entries to strings
df['text'] = df['text'].astype(str)


In [2]:
# Define a simple vocabulary
vocab = {'<pad>': 0, '<unk>': 1}
for text in df['text']:
    for word in text.split():
        if word not in vocab:
            vocab[word] = len(vocab)


In [3]:
class MenuDataset(Dataset):
    def __init__(self, texts, vocab, max_length):
        self.texts = texts
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        tokens = self.tokenize(text)
        return {
            'src': tokens,  # Use 'src' instead of 'tokens' for compatibility with training function
            'tgt_input': tokens[:-1] + [self.vocab['<pad>']],  # Input to the decoder
            'tgt_output': tokens[1:] + [self.vocab['<pad>']]  # Output from the decoder
        }

    def tokenize(self, text):
        tokens = text.split()
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]
        token_ids = [self.vocab.get(token, self.vocab['<unk>']) for token in tokens]
        token_ids += [self.vocab['<pad>']] * (self.max_length - len(token_ids))
        return token_ids

def collate_fn(batch):
    src = [item['src'] for item in batch]
    tgt_input = [item['tgt_input'] for item in batch]
    tgt_output = [item['tgt_output'] for item in batch]

    src = torch.tensor(src, dtype=torch.long)
    tgt_input = torch.tensor(tgt_input, dtype=torch.long)
    tgt_output = torch.tensor(tgt_output, dtype=torch.long)

    return {
        'src': src,
        'tgt_input': tgt_input,
        'tgt_output': tgt_output
    }

# Create the dataset and dataloader
max_length = 512
dataset = MenuDataset(df['text'].tolist(), vocab, max_length)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)


In [4]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.d_model = d_model  # Save d_model for later use
        self.dropout = nn.Dropout(p=0.1)
        
        # Compute positional encodings once in log space
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x * math.sqrt(self.d_model)
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_encoder_layers, num_decoder_layers, dim_feedforward, max_len=512):
        super(TransformerModel, self).__init__()
        
        self.d_model = d_model  # Save d_model for later use
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward
        )
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt_input, src_mask, tgt_mask, memory_mask):
        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        tgt_input = self.embedding(tgt_input) * math.sqrt(self.d_model)
        tgt_input = self.pos_encoder(tgt_input)
        output = self.transformer(src, tgt_input, src_mask, tgt_mask, memory_mask)
        output = self.fc_out(output)
        return output


In [5]:
def generate_square_subsequent_mask(size):
    mask = (torch.triu(torch.ones(size, size)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

def create_mask(src, tgt_input):
    src_seq_len = src.size(0)
    tgt_seq_len = tgt_input.size(0)
    
    src_mask = generate_square_subsequent_mask(src_seq_len)
    tgt_mask = generate_square_subsequent_mask(tgt_seq_len)
    
    src_padding_mask = (src == 0).transpose(0, 1)
    tgt_padding_mask = (tgt_input == 0).transpose(0, 1)
    
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask


In [6]:
def train(model, dataloader, criterion, optimizer, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        for batch in dataloader:
            src = batch['src'].transpose(0, 1)
            tgt_input = batch['tgt_input'].transpose(0, 1)
            tgt_output = batch['tgt_output'].transpose(0, 1)
            
            src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input)
            
            optimizer.zero_grad()
            output = model(src, tgt_input, src_mask, tgt_mask, None)
            loss = criterion(output.view(-1, output.size(-1)), tgt_output.contiguous().view(-1))
            loss.backward()
            optimizer.step()
            
            print(f"Epoch {epoch+1}, Loss: {loss.item()}")


In [7]:
# Hyperparameters
vocab_size = len(vocab)
d_model = 512
nhead = 8
num_encoder_layers = 6
num_decoder_layers = 6
dim_feedforward = 2048
num_epochs = 10
learning_rate = 0.001

# Instantiate the model, criterion, and optimizer
model = TransformerModel(vocab_size, d_model, nhead, num_encoder_layers, num_decoder_layers, dim_feedforward)
criterion = nn.CrossEntropyLoss(ignore_index=vocab['<pad>'])
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


In [11]:
# Example usage of training
train(model, dataloader, criterion, optimizer, num_epochs)


Epoch 1, Loss: 6.870450973510742
Epoch 1, Loss: 6.655766487121582
Epoch 1, Loss: 7.451169013977051
Epoch 1, Loss: 7.2356109619140625
Epoch 1, Loss: 7.22504186630249
Epoch 1, Loss: 7.055228233337402
Epoch 1, Loss: 7.660529136657715
Epoch 1, Loss: 6.993017196655273
Epoch 1, Loss: 7.379428863525391
Epoch 1, Loss: 7.362977981567383
Epoch 1, Loss: 7.140920639038086
Epoch 1, Loss: 7.179027080535889
Epoch 1, Loss: 6.923509120941162
Epoch 1, Loss: 6.735969066619873
Epoch 1, Loss: 7.107647895812988
Epoch 1, Loss: 7.470365047454834
Epoch 1, Loss: 7.2226057052612305
Epoch 1, Loss: 6.9944682121276855
Epoch 1, Loss: 6.996205806732178
Epoch 1, Loss: 7.336209774017334
Epoch 1, Loss: 6.901638507843018
Epoch 1, Loss: 7.339893341064453
Epoch 1, Loss: 7.071323871612549
Epoch 1, Loss: 6.926355361938477
Epoch 1, Loss: 6.82535982131958
Epoch 1, Loss: 7.080330848693848
Epoch 1, Loss: 6.909735679626465
Epoch 1, Loss: 7.033660888671875
Epoch 1, Loss: 6.768629550933838
Epoch 1, Loss: 6.748590469360352
Epoch 1, 

KeyboardInterrupt: 

In [12]:
def generate_menu(prompt, model, vocab, max_length=50):
    model.eval()
    src = torch.tensor([dataset.tokenize(prompt)], dtype=torch.long).to(next(model.parameters()).device)
    src = src.transpose(0, 1)  # Adjust dimensions for model
    src_mask, _, _, _ = create_mask(src, src)  # Create mask for source

    memory = model.transformer.encoder(model.embedding(src) * math.sqrt(model.d_model), src_mask)
    ys = torch.ones(1, 1).fill_(vocab['<pad>']).type(torch.long).to(src.device)  # Initial target token

    for i in range(max_length-1):
        tgt_mask = generate_square_subsequent_mask(ys.size(0)).to(ys.device)
        out = model.transformer.decoder(model.embedding(ys) * math.sqrt(model.d_model), memory, tgt_mask)
        out = model.fc_out(out)
        prob = out[-1, :].squeeze().div(1.0).exp()
        _, next_word = torch.max(prob, dim=0)
        ys = torch.cat([ys, next_word.unsqueeze(0).unsqueeze(1)], dim=0)
        if next_word == vocab['<pad>']:
            break
    return ' '.join([list(vocab.keys())[i] for i in ys.squeeze().tolist()])

# Example usage
prompt = "Create a menu for a Mexican restaurant"
menu = generate_menu(prompt, model, vocab)
print(menu)


<pad> the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the
